# RAG with TOC-Guided Chunking

This notebook mirrors the cleaned `RAG_with_TOC.py` pipeline.

Pipeline:
1. Load the PDF TOC, trim it to the content range, and enrich each entry with section hierarchy.
2. Normalize extracted text before heading matching.
3. Flatten PyMuPDF page text into line records with `text`, `font_size`, and `top`.
4. Match headings with pragmatic fallbacks for subtitle-only matches, split heading lines, and minor extraction glitches.
5. Extract section text between successive TOC headings while removing continuation-page running headers, listing continuation markers, and numeric boundary noise.
6. Build overlapping chunks, write outlier reports, and save retrieval-ready artifacts.

Run the cells in order. The debug cells verify:
- normalization and tolerant heading comparison
- TOC filtering and hierarchical enrichment
- page-line extraction, heading lookup, and section boundary slicing
- chunk statistics and outlier reporting
- running-header stripping on continuation pages without altering chapter-opening pages

In [1]:
import json
import logging
import re
import sys
import unicodedata
from dataclasses import dataclass
from typing import Dict, List, Optional

import fitz
import tiktoken

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PDF_PATH = "clean-code.pdf"
TOC_START = "Foreword"
TOC_END = "Tutorial: Full Code Examples"
BODY_FONT_SIZE = 10
MIN_SECTION_TOKENS = 10
MIN_CHUNK_TOKENS = 40
SENTENCES_PER_CHUNK = 15
MAX_CHUNK_TOKENS = 500
OVERLAP_SENTENCES = 2
SMALL_CHUNKS_REPORT_PATH = "small_chunks.txt"
BIG_CHUNKS_REPORT_PATH = "big_chunks.txt"
FAISS_INDEX_PATH = "chunks.index"
CHUNKS_METADATA_PATH = "chunks_metadata.json"

_enc = tiktoken.get_encoding("cl100k_base")

## Core data and text helpers

`TocEntry` stores the TOC level together with resolved main/section/subsection context.

`TextChunk` stores the final chunk text plus the page and section metadata that later gets written to retrieval metadata.

The helpers in this cell cover four jobs:
- text normalization and tolerant heading comparison
- continuation-page running-header removal
- listing continuation cleanup
- TOC filtering and hierarchical enrichment

In [2]:
@dataclass
class TocEntry:
    level: int
    title: str
    page: int
    main_section: Optional[str]
    section: Optional[str]
    subsection: Optional[str]


@dataclass
class TextChunk:
    content: str
    page_number: int
    main_section: Optional[str]
    section: Optional[str]
    subsection: Optional[str]
    chunk_index: int = 0


def count_tokens(text: str) -> int:
    return len(_enc.encode(text))


def count_sentences(text: str) -> int:
    return len(split_sentences(text))


def normalize_text(text: str) -> str:
    """Normalize Unicode and quotes once before any matching or extraction."""
    text = unicodedata.normalize("NFKC", text)
    return text.replace("\u2018", "'").replace("\u2019", "'").strip()


def relax_heading_match(text: str) -> str:
    """Loosen heading text just enough for known PDF extraction glitches.

    Example: this book's TOC contains "Congurable" while the page heading is
    "Configurable". Dropping common "fi"-style pairs on both sides keeps the
    comparison pragmatic without changing the extracted section text itself.
    """
    for pair in ("ffi", "ffl", "fi", "fl", "ff"):
        text = text.replace(pair, "")
    return text


def headings_match(left: str, right: str) -> bool:
    return relax_heading_match(left) == relax_heading_match(right)


def remove_running_header(page_lines: List[Dict], body_font_size: int = BODY_FONT_SIZE) -> List[Dict]:
    """Drop the two-line running header used on continuation pages in this PDF.

    In practice the leaked header has a very regular shape:
    - a page number line in the top margin
    - a short title line beside it, also in body-sized text
    - a clear vertical gap before the real page content begins

    Chapter-opening pages such as "Meaningful Names" do not match that shape,
    because their visible headings start much lower on the page and use larger fonts.
    """
    if len(page_lines) < 3:
        return page_lines

    first_line, second_line, third_line = page_lines[:3]
    is_page_number = bool(re.fullmatch(r"[ivxlcdm]+|\d+", first_line["text"].lower()))
    looks_like_title = (
        len(second_line["text"]) <= 60
        and len(second_line["text"].split()) <= 8
    )
    has_top_margin_layout = (
        first_line["top"] <= 50
        and second_line["top"] <= 50
        and third_line["top"] - second_line["top"] >= 20
    )
    has_body_sized_header = (
        first_line["font_size"] <= body_font_size
        and second_line["font_size"] <= body_font_size
    )

    if is_page_number and looks_like_title and has_top_margin_layout and has_body_sized_header:
        return page_lines[2:]

    return page_lines


def remove_listing_continuation_lines(page_lines: List[Dict]) -> List[Dict]:
    """Drop listing continuation markers and their attached filename labels."""
    cleaned: List[Dict] = []
    skip_next_file_label = False

    for line in page_lines:
        if re.fullmatch(r"Listing\s+[\w.-]+\s+\(continued\)", line["text"]):
            skip_next_file_label = True
            continue

        if skip_next_file_label and re.fullmatch(
            r"[\w.-]+\.(java|kt|py|js|ts|cpp|c|cs)",
            line["text"],
        ):
            skip_next_file_label = False
            continue

        skip_next_file_label = False
        cleaned.append(line)

    return cleaned


def filter_toc(raw_toc: List[list], start_title: str, end_title: str) -> List[list]:
    start_matches = [index for index, entry in enumerate(raw_toc) if entry[1] == start_title]
    end_matches = [index for index, entry in enumerate(raw_toc) if entry[1] == end_title]

    if len(start_matches) != 1:
        raise ValueError(
            f"Expected exactly 1 match for TOC_START='{start_title}', found {len(start_matches)}"
        )
    if len(end_matches) != 1:
        raise ValueError(
            f"Expected exactly 1 match for TOC_END='{end_title}', found {len(end_matches)}"
        )

    start_index = start_matches[0]
    end_index = end_matches[0]
    return raw_toc[start_index:end_index]


def build_enriched_toc(raw_toc: List[list]) -> List[TocEntry]:
    entries: List[TocEntry] = []
    current_main = None
    current_section = None

    for level, title, page in raw_toc:
        if level == 1:
            current_main = title
            current_section = None
            main_section = title
            section = None
            subsection = None
        elif level == 2:
            current_section = title
            main_section = current_main
            section = title
            subsection = None
        else:
            main_section = current_main
            section = current_section
            subsection = title

        entries.append(
            TocEntry(
                level=level,
                title=title,
                page=page,
                main_section=main_section,
                section=section,
                subsection=subsection,
            )
        )

    return entries

In [3]:
normalization_examples = [
    ("curly quotes", "It's clean.", "It’s clean."),
    (
        "TOC typo vs page heading",
        "G35: Keep Congurable Data at High Levels",
        "G35: Keep Configurable Data at High Levels",
    ),
]

for label, left, right in normalization_examples:
    print(label)
    print(f"  left raw        : {left!r}")
    print(f"  right raw       : {right!r}")
    print(f"  left normalized : {normalize_text(left)!r}")
    print(f"  right normalized: {normalize_text(right)!r}")
    print(f"  relaxed match   : {headings_match(normalize_text(left), normalize_text(right))}")
    print()

curly quotes
  left raw        : "It's clean."
  right raw       : 'It’s clean.'
  left normalized : "It's clean."
  right normalized: "It's clean."
  relaxed match   : True

TOC typo vs page heading
  left raw        : 'G35: Keep Congurable Data at High Levels'
  right raw       : 'G35: Keep Configurable Data at High Levels'
  left normalized : 'G35: Keep Congurable Data at High Levels'
  right normalized: 'G35: Keep Configurable Data at High Levels'
  relaxed match   : True



## Load the PDF and parse the TOC

This step opens the PDF, reads the raw TOC, trims it to `[TOC_START, TOC_END)`, and converts the remaining entries into `TocEntry` records with section hierarchy attached.

`TOC_END` is used only as the exclusive boundary marker, so it is not parsed as content to extract.

In [4]:
doc = fitz.open(PDF_PATH)
raw_toc = doc.get_toc()
filtered_toc = filter_toc(raw_toc, TOC_START, TOC_END)
content_toc = build_enriched_toc(filtered_toc)

print(f"Raw TOC entries: {len(raw_toc)}")
print(f"Filtered TOC entries: {len(filtered_toc)}")
print(f"First filtered entry: {filtered_toc[0]}")
print(f"Last filtered entry: {filtered_toc[-1]}")
print(f"First enriched entry: {content_toc[0]}")
print(f"Last enriched entry: {content_toc[-1]}")
print()

for entry in content_toc[:5]:
    indent = "    " * (entry.level - 1)
    print(f"{indent}[L{entry.level}] p.{entry.page:<4} {entry.title}")

assert content_toc[0].title == TOC_START
assert content_toc[-1].title != TOC_END

Raw TOC entries: 384
Filtered TOC entries: 351
First filtered entry: [1, 'Foreword', 20]
Last filtered entry: [2, 'Conclusion', 373]
First enriched entry: TocEntry(level=1, title='Foreword', page=20, main_section='Foreword', section=None, subsection=None)
Last enriched entry: TocEntry(level=2, title='Conclusion', page=373, main_section='Appendix A: Concurrency II', section='Conclusion', subsection=None)

[L1] p.20   Foreword
[L1] p.26   Introduction
[L1] p.30   On the Cover
[L1] p.32   Chapter 1: Clean Code
    [L2] p.33   There Will Be Code


## Page line extraction and heading lookup

`get_page_lines()` flattens PyMuPDF's nested `dict` output into reading-order lines with `text`, `font_size`, and `top`.

That information drives several later heuristics:
- `font_size` separates headings from body text
- `top` lets the pipeline remove continuation-page running headers
- listing continuation lines are stripped before text extraction
- end-page numeric boundary lines are trimmed before the next section begins

A heading can match in four practical ways:
- exact line match
- text after `:`
- two consecutive heading lines joined together
- relaxed comparison for minor ligature or extraction glitches

In [5]:
def get_page_lines(doc: fitz.Document, page_1idx: int, strip_running_header: bool = True) -> List[Dict]:
    """Return reading-order lines as dictionaries with text, font size, and top position.

    PyMuPDF page.get_text("dict") returns nested page data:
    block -> line -> span. We flatten that structure to one entry per visible
    line, for example: {"text": "Clean Code", "font_size": 24, "top": 202.7}.

    font_size is intentionally kept because heading detection uses it to
    separate section titles from normal body text. top is kept so a small,
    isolated heuristic can drop continuation-page running headers.
    """
    page_lines: List[Dict] = []
    page = doc[page_1idx - 1]

    for block in page.get_text("dict")["blocks"]:
        if block.get("type") != 0:
            # type is 0 for text blocks, 1 for images, 2 for drawings, etc. We only want text.
            continue

        for line in block["lines"]:
            text = normalize_text("".join(span["text"] for span in line["spans"]))
            font_size = round(max((span["size"] for span in line["spans"]), default=0))
            top = min((span["bbox"][1] for span in line["spans"]), default=0)
            if text:
                page_lines.append({"text": text, "font_size": font_size, "top": top})

    if strip_running_header:
        page_lines = remove_running_header(page_lines, BODY_FONT_SIZE)
        page_lines = remove_listing_continuation_lines(page_lines)
        return page_lines

    return page_lines


def find_heading(page_lines: List[Dict], title: str, body_font_size: int) -> int:
    """Return the index of the heading line that starts a TOC entry.

    Two pragmatic fallbacks are used:
    1. If the TOC says "Chapter 1: Clean Code", the page may only show
       "Clean Code", so we also try the text after ':'.
    2. Some headings are split across two lines with the same font size,
       for example "Make Meaningful" + "Distinctions".
    """
    normalized_title = normalize_text(title)
    candidates = [normalized_title]
    if ":" in normalized_title:
        candidates.append(normalized_title.split(":", 1)[1].strip())

    heading_lines = [
        (index, line)
        for index, line in enumerate(page_lines)
        if line["font_size"] > body_font_size
    ]

    for search in candidates:
        for position, (index, line) in enumerate(heading_lines):
            text = line["text"].rstrip("0123456789")
            if headings_match(text, search):
                return index

            if search.startswith(text) and position + 1 < len(heading_lines):
                _, next_line = heading_lines[position + 1]
                combined_text = f"{text} {next_line['text']}"
                if next_line["font_size"] == line["font_size"] and headings_match(combined_text, search):
                    return index

    raise ValueError(f"No heading found for '{title}'")


def merge_line_text(parts: List[str], text: str) -> None:
    """Append a line while preserving punctuation-led continuations.

    Some code listings wrap a chained call onto the next visual line, for
    example `newPageContent` followed by `.append(...)`. Joining every line
    with a space turns that into `newPageContent .append(...)`, so lines that
    start with punctuation are attached directly to the previous part.
    """
    if parts and text[:1] in ".,;:)]}":
        parts[-1] += text
        return

    parts.append(text)


def trim_end_page_boundary_lines(lines: List[Dict]) -> List[Dict]:
    """Drop numeric-only boundary lines that belong to the next page heading.

    Chapter-opening pages can contribute a footer page number and a chapter number
    just before the next heading match. Those lines are not content from the
    current section, so trim them from the end-page slice only.
    """
    trimmed = list(lines)
    while trimmed and re.fullmatch(r"\d+", trimmed[-1]["text"]):
        trimmed.pop()
    return trimmed


def extract_section_text(
    doc: fitz.Document,
    start_page: int,
    start_heading_idx: int,
    end_page: int,
    end_heading_idx: int,
    strip_running_header: bool = True,
) -> str:
    """Extract all text between two heading positions, excluding both headings."""
    parts: List[str] = []

    for page_num in range(start_page, end_page + 1):
        lines = get_page_lines(doc, page_num, strip_running_header=strip_running_header)
        if page_num == start_page and page_num == end_page:
            selected = lines[start_heading_idx + 1:end_heading_idx]
        elif page_num == start_page:
            selected = lines[start_heading_idx + 1:]
        elif page_num == end_page:
            selected = trim_end_page_boundary_lines(lines[:end_heading_idx])
        else:
            selected = lines

        for line in selected:
            merge_line_text(parts, line["text"])

    return " ".join(parts)

In [6]:
entry = next(item for item in content_toc if item.title == "Chapter 1: Clean Code")
page_lines = get_page_lines(doc, entry.page)

print(f"Page: {entry.page}")
print(f"Body font size threshold: {BODY_FONT_SIZE}")
print()
for index, line in enumerate(page_lines):
    marker = " <<< heading candidate" if line["font_size"] > BODY_FONT_SIZE else ""
    print(f"[{index:>2}] top={line['top']:>6.1f} size={line['font_size']:>2} text={line['text']!r}{marker}")

Page: 32
Body font size threshold: 10

[ 0] top= 621.9 size=10 text='1'
[ 1] top= 102.4 size=48 text='1' <<< heading candidate
[ 2] top= 202.7 size=24 text='Clean Code' <<< heading candidate
[ 3] top= 559.4 size=10 text='You are reading this book for two reasons. First, you are a programmer. Second, you want'
[ 4] top= 571.4 size=10 text='to be a better programmer. Good. We need better programmers.'


## Debug heading matching

These examples exercise the matcher paths used in practice:
- colon fallback: `Chapter 1: Clean Code` -> `Clean Code`
- multi-line heading join: `Make Meaningful` + `Distinctions`
- appendix-style split title: `Appendix A` + `Concurrency II`
- relaxed typo tolerance: `Congurable` vs `Configurable`

In [7]:
debug_titles = [
    "Chapter 1: Clean Code",
    "Make Meaningful Distinctions",
    "Appendix A: Concurrency II",
    "G35: Keep Congurable Data at High Levels",
]

for title in debug_titles:
    entry = next(item for item in content_toc if item.title == title)
    page_lines = get_page_lines(doc, entry.page)
    heading_index = find_heading(page_lines, entry.title, BODY_FONT_SIZE)

    normalized_title = normalize_text(entry.title)
    candidates = [normalized_title]
    if ":" in normalized_title:
        candidates.append(normalized_title.split(":", 1)[1].strip())

    matched_line = page_lines[heading_index]["text"]
    next_line_text = page_lines[heading_index + 1]["text"] if heading_index + 1 < len(page_lines) else ""
    combined_text = f"{matched_line} {next_line_text}".strip()
    matched_candidate = next(
        (
            candidate
            for candidate in candidates
            if headings_match(matched_line, candidate) or headings_match(combined_text, candidate)
        ),
        None,
    )

    print(f"Title: {entry.title!r}")
    print(f"  page: {entry.page}")
    print(f"  candidates: {candidates}")
    print(f"  matched line: {matched_line!r}")
    print(f"  combined line preview: {combined_text!r}")
    print(f"  matched candidate: {matched_candidate!r}")
    print(f"  used fallback: {matched_candidate != normalized_title}")
    print()

Title: 'Chapter 1: Clean Code'
  page: 32
  candidates: ['Chapter 1: Clean Code', 'Clean Code']
  matched line: 'Clean Code'
  combined line preview: 'Clean Code You are reading this book for two reasons. First, you are a programmer. Second, you want'
  matched candidate: 'Clean Code'
  used fallback: True

Title: 'Make Meaningful Distinctions'
  page: 51
  candidates: ['Make Meaningful Distinctions']
  matched line: 'Make Meaningful'
  combined line preview: 'Make Meaningful Distinctions'
  matched candidate: 'Make Meaningful Distinctions'
  used fallback: False

Title: 'Appendix A: Concurrency II'
  page: 348
  candidates: ['Appendix A: Concurrency II', 'Concurrency II']
  matched line: 'Concurrency II'
  combined line preview: 'Concurrency II by Brett L. Schuchert'
  matched candidate: 'Concurrency II'
  used fallback: True

Title: 'G35: Keep Congurable Data at High Levels'
  page: 337
  candidates: ['G35: Keep Congurable Data at High Levels', 'Keep Congurable Data at High Levels']


## Chunking and full parse

The full parser walks TOC entries in order, extracts text between successive headings, skips sections under the minimum section token count, and then chunks each section with the same rules as the script:
- section boundaries come from successive TOC headings
- chunks are grown sentence by sentence until either the sentence limit or token limit would be exceeded
- the next chunk reuses the last 2 sentences from the previous chunk
- a sentence over 500 tokens is dropped and written to the big-outlier report
- a chunk under 40 tokens is dropped and written to the small-outlier report
- a chunk over 500 tokens is also treated as a big outlier, even though the chunker is designed to avoid that

In [8]:
def split_sentences(text: str) -> List[str]:
    """Split text into pragmatic sentence units for chunking.

    The PDF extraction already normalizes lines and joins them with spaces, so a
    lightweight punctuation-based split is enough for this pipeline.
    """
    normalized = re.sub(r"\s+", " ", text).strip()
    if not normalized:
        return []

    return [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", normalized) if sentence.strip()]


def make_outlier_record(
    entry: TocEntry,
    content: str,
    reason: str,
    chunk_index: Optional[int] = None,
) -> Dict:
    return {
        "reason": reason,
        "main_section": entry.main_section,
        "section": entry.section,
        "subsection": entry.subsection,
        "page_number": entry.page,
        "chunk_index": chunk_index,
        "token_count": count_tokens(content),
        "sentence_count": count_sentences(content),
        "content": content,
    }


def write_outlier_report(path: str, title: str, outliers: List[Dict]) -> None:
    with open(path, "w", encoding="utf-8") as file:
        file.write(f"{title}\n")
        file.write(f"count={len(outliers)}\n\n")

        if not outliers:
            file.write("No outlier chunks found.\n")
            return

        for index, outlier in enumerate(outliers, start=1):
            file.write(f"=== Outlier {index} ===\n")
            file.write(f"reason={outlier['reason']}\n")
            file.write(f"main_section={outlier['main_section']!r}\n")
            file.write(f"section={outlier['section']!r}\n")
            file.write(f"subsection={outlier['subsection']!r}\n")
            file.write(f"page_number={outlier['page_number']}\n")
            file.write(f"chunk_index={outlier['chunk_index']}\n")
            file.write(f"sentence_count={outlier['sentence_count']}\n")
            file.write(f"token_count={outlier['token_count']}\n")
            file.write("content:\n")
            file.write(f"{outlier['content']}\n\n")


def build_sentence_limited_chunks(
    sentences: List[str],
    max_sentences: int = SENTENCES_PER_CHUNK,
    max_tokens: int = MAX_CHUNK_TOKENS,
    overlap_sentences: int = OVERLAP_SENTENCES,
) -> Dict[str, List[str]]:
    """Build chunks that respect both sentence count and token count limits.

    Sentences are added one by one until adding another sentence would exceed
    either limit. The next chunk then starts with a 2-sentence overlap.
    Oversized sentences are dropped instead of being force-split.
    """
    if not sentences:
        return {"chunks": [], "dropped_oversized_sentences": []}
    if overlap_sentences >= max_sentences:
        raise ValueError("overlap_sentences must be smaller than max_sentences")

    chunks: List[str] = []
    dropped_oversized_sentences: List[str] = []
    start = 0

    while start < len(sentences):
        chunk_sentences: List[str] = []
        oversized_sentence_consumed = False

        for sentence in sentences[start:]:
            sentence_tokens = count_tokens(sentence)
            if sentence_tokens > max_tokens:
                if chunk_sentences:
                    break
                dropped_oversized_sentences.append(sentence)
                oversized_sentence_consumed = True
                break

            candidate_sentences = chunk_sentences + [sentence]
            candidate_text = " ".join(candidate_sentences)
            if chunk_sentences and (
                len(candidate_sentences) > max_sentences or count_tokens(candidate_text) > max_tokens
            ):
                break

            chunk_sentences = candidate_sentences

            if len(chunk_sentences) >= max_sentences:
                break

        if oversized_sentence_consumed:
            start += 1
            continue

        if not chunk_sentences:
            start += 1
            continue

        chunks.append(" ".join(chunk_sentences))
        if start + len(chunk_sentences) >= len(sentences):
            break

        step = max(1, len(chunk_sentences) - overlap_sentences)
        start += step

    return {"chunks": chunks, "dropped_oversized_sentences": dropped_oversized_sentences}


def chunk_text(
    text: str,
    sentences_per_chunk: int = SENTENCES_PER_CHUNK,
    max_chunk_tokens: int = MAX_CHUNK_TOKENS,
    overlap_sentences: int = OVERLAP_SENTENCES,
) -> Dict[str, List[str]]:
    """Split text into chunks that respect both sentence and token limits."""
    sentences = split_sentences(text)
    return build_sentence_limited_chunks(sentences, sentences_per_chunk, max_chunk_tokens, overlap_sentences)


def parse_pages(
    doc: fitz.Document,
    content_toc: List[TocEntry],
    max_pages: Optional[int] = None,
) -> List[TextChunk]:
    """Parse the book by TOC boundaries and return sentence-based chunks."""
    first_page = content_toc[0].page
    last_page = content_toc[-1].page
    if max_pages is not None:
        last_page = min(last_page, first_page + max_pages - 1)

    active_entries = [entry for entry in content_toc if entry.page <= last_page]
    logger.info("Processing %d TOC entries", len(active_entries))

    sections = []
    for index, entry in enumerate(active_entries):
        start_lines = get_page_lines(doc, entry.page)
        start_idx = find_heading(start_lines, entry.title, BODY_FONT_SIZE)

        if index + 1 < len(active_entries):
            next_entry = active_entries[index + 1]
            end_lines = get_page_lines(doc, next_entry.page)
            end_idx = find_heading(end_lines, next_entry.title, BODY_FONT_SIZE)
            end_page = next_entry.page
        else:
            end_page = last_page
            end_idx = len(get_page_lines(doc, last_page))

        text = extract_section_text(doc, entry.page, start_idx, end_page, end_idx).strip()
        if text and count_tokens(text) >= MIN_SECTION_TOKENS:
            sections.append({"text": text, "entry": entry})

    logger.info("Sections extracted: %d", len(sections))

    all_chunks: List[TextChunk] = []
    small_outliers: List[Dict] = []
    big_outliers: List[Dict] = []
    for section in sections:
        entry = section["entry"]
        chunk_result = chunk_text(section["text"])

        for dropped_sentence in chunk_result["dropped_oversized_sentences"]:
            big_outliers.append(make_outlier_record(entry, dropped_sentence, "dropped_oversized_sentence"))

        for chunk_index, chunk_str in enumerate(chunk_result["chunks"]):
            chunk = TextChunk(
                content=chunk_str,
                page_number=entry.page,
                main_section=entry.main_section,
                section=entry.section,
                subsection=entry.subsection,
                chunk_index=chunk_index,
            )
            token_count = count_tokens(chunk.content)
            if token_count < MIN_CHUNK_TOKENS:
                small_outliers.append(make_outlier_record(entry, chunk.content, "dropped_small_chunk", chunk_index))
                continue
            if token_count > MAX_CHUNK_TOKENS:
                big_outliers.append(make_outlier_record(entry, chunk.content, "dropped_large_chunk", chunk_index))
                continue
            all_chunks.append(chunk)

    write_outlier_report(SMALL_CHUNKS_REPORT_PATH, "Small chunk outliers", small_outliers)
    write_outlier_report(BIG_CHUNKS_REPORT_PATH, "Big chunk outliers", big_outliers)

    logger.info("Chunks created: %d", len(all_chunks))
    logger.info("Small chunk outliers written: %d", len(small_outliers))
    logger.info("Big chunk outliers written: %d", len(big_outliers))
    return all_chunks

## Full run

Run the end-to-end parse, then verify the final constraints and inspect the resulting chunk metadata:
- no kept chunk exceeds 15 sentences
- no kept chunk falls outside the 40 to 500 token window
- both outlier files are overwritten on each run
- the printed sample confirms the stored page and section metadata for retrieval

In [9]:
chunks = parse_pages(doc, content_toc)
sentence_counts = [count_sentences(chunk.content) for chunk in chunks]
token_counts = [count_tokens(chunk.content) for chunk in chunks]

print(f"Total chunks: {len(chunks)}")
print(
    f"Sentence stats: min={min(sentence_counts)}, avg={sum(sentence_counts) // len(sentence_counts)}, max={max(sentence_counts)}"
)
print(
    f"Chunks over {SENTENCES_PER_CHUNK} sentences: {sum(1 for count in sentence_counts if count > SENTENCES_PER_CHUNK)}"
)
print(
    f"Chunks under {MIN_CHUNK_TOKENS} tokens: {sum(1 for count in token_counts if count < MIN_CHUNK_TOKENS)}"
)
print(
    f"Chunks over {MAX_CHUNK_TOKENS} tokens: {sum(1 for count in token_counts if count > MAX_CHUNK_TOKENS)}"
)
print(f"Token stats: min={min(token_counts)}, avg={sum(token_counts) // len(token_counts)}, max={max(token_counts)}")
print(f"Outlier reports: {SMALL_CHUNKS_REPORT_PATH}, {BIG_CHUNKS_REPORT_PATH}")

assert all(count <= SENTENCES_PER_CHUNK for count in sentence_counts)
assert all(MIN_CHUNK_TOKENS <= count <= MAX_CHUNK_TOKENS for count in token_counts)

first_chunk = chunks[0]
print()
print("First chunk metadata:")
print(f"  page={first_chunk.page_number}")
print(f"  main_section={first_chunk.main_section!r}")
print(f"  section={first_chunk.section!r}")
print(f"  chunk_index={first_chunk.chunk_index}")
print(f"  sentences={sentence_counts[0]}")
print(f"  tokens={token_counts[0]}")
print(f"  preview={first_chunk.content[:20000]!r}")

INFO: Processing 351 TOC entries
INFO: Sections extracted: 341
INFO: Chunks created: 582
INFO: Small chunk outliers written: 32
INFO: Big chunk outliers written: 11


Total chunks: 582
Sentence stats: min=1, avg=10, max=15
Chunks over 15 sentences: 0
Chunks under 40 tokens: 0
Chunks over 500 tokens: 0
Token stats: min=41, avg=243, max=499
Outlier reports: small_chunks.txt, big_chunks.txt

First chunk metadata:
  page=20
  main_section='Foreword'
  section=None
  chunk_index=0
  sentences=15
  tokens=327
  preview='One of our favorite candies here in Denmark is Ga-Jol, whose strong licorice vapors are a perfect complement to our damp and often chilly weather. Part of the charm of Ga-Jol to us Danes is the wise or witty sayings printed on the flap of every box top. I bought a two- pack of the delicacy this morning and found that it bore this old Danish saw: Ærlighed i små ting er ikke nogen lille ting. “Honesty in small things is not a small thing.” It was a good omen consistent with what I already wanted to say here. Small things matter. This is a book about humble concerns whose value is nonetheless far from small. God is in the details, said the ar

In [10]:
min_token_index = min(range(len(chunks)), key=lambda index: token_counts[index])
max_token_index = max(range(len(chunks)), key=lambda index: token_counts[index])

min_chunk = chunks[min_token_index]
max_chunk = chunks[max_token_index]

print("Chunk token debug:")
print(f"  total chunks={len(chunks)}")
print(f"  min tokens={token_counts[min_token_index]} at chunk #{min_token_index}")
print(f"  max tokens={token_counts[max_token_index]} at chunk #{max_token_index}")
print()
print("Min-token chunk:")
print(f"  page={min_chunk.page_number}, section={min_chunk.section!r}, sentences={sentence_counts[min_token_index]}")
print(f"  preview={min_chunk.content[:200]!r}")
print()
print("Max-token chunk:")
print(f"  page={max_chunk.page_number}, section={max_chunk.section!r}, sentences={sentence_counts[max_token_index]}")
print(f"  preview={max_chunk.content[:200]!r}")
print()
for report_path in (SMALL_CHUNKS_REPORT_PATH, BIG_CHUNKS_REPORT_PATH):
    print(f"Report preview: {report_path}")
    with open(report_path, "r", encoding="utf-8") as file:
        for line in file.readlines()[:8]:
            print(f"  {line.rstrip()}")
    print()

Chunk token debug:
  total chunks=582
  min tokens=41 at chunk #354
  max tokens=499 at chunk #87

Min-token chunk:
  page=217, section='Testing Threaded Code', sentences=5
  preview='• Make your threaded code pluggable. • Make your threaded code tunable. • Run with more threads than processors. • Run on different platforms. • Instrument your code to try and force failures.'

Max-token chunk:
  page=62, section=None, sentences=13
  preview='In the early days of programming we composed our systems of routines and subroutines. Then, in the era of Fortran and PL/1 we composed our systems of programs, subprograms, and functions. Nowadays onl'

Report preview: small_chunks.txt
  Small chunk outliers
  count=32
  
  === Outlier 1 ===
  reason=dropped_small_chunk
  main_section='Chapter 4: Comments'
  section='Bad Comments'
  subsection=None

Report preview: big_chunks.txt
  Big chunk outliers
  count=11
  
  === Outlier 1 ===
  reason=dropped_oversized_sentence
  main_section='Chapter 3: Fun

## Debug running-header removal

These checks use a real continuation-page leak from `Foreword` and a chapter-opening counterexample from `Chapter 2: Meaningful Names`.

The first cell shows the top lines before and after stripping, then confirms the leaked running-header fragment disappears from extracted section text.
The second cell verifies that a chapter-opening page is left unchanged because it does not match the continuation-header shape.

In [11]:
foreword_entry = next(item for item in content_toc if item.title == "Foreword")
introduction_entry = next(item for item in content_toc if item.title == "Introduction")
running_header_page = foreword_entry.page + 1

raw_page_lines = get_page_lines(doc, running_header_page, strip_running_header=False)
clean_page_lines = get_page_lines(doc, running_header_page)

print(f"Running-header page: {running_header_page}")
print("Raw top lines:")
for line in raw_page_lines[:3]:
    print(f"  top={line['top']:>6.1f} size={line['font_size']:>2} text={line['text']!r}")

print()
print("After remove_running_header():")
for line in clean_page_lines[:3]:
    print(f"  top={line['top']:>6.1f} size={line['font_size']:>2} text={line['text']!r}")

raw_start_lines = get_page_lines(doc, foreword_entry.page, strip_running_header=False)
raw_end_lines = get_page_lines(doc, introduction_entry.page, strip_running_header=False)
clean_start_lines = get_page_lines(doc, foreword_entry.page)
clean_end_lines = get_page_lines(doc, introduction_entry.page)

raw_text = extract_section_text(
    doc,
    foreword_entry.page,
    find_heading(raw_start_lines, foreword_entry.title, BODY_FONT_SIZE),
    introduction_entry.page,
    find_heading(raw_end_lines, introduction_entry.title, BODY_FONT_SIZE),
    strip_running_header=False,
 )
clean_text = extract_section_text(
    doc,
    foreword_entry.page,
    find_heading(clean_start_lines, foreword_entry.title, BODY_FONT_SIZE),
    introduction_entry.page,
    find_heading(clean_end_lines, introduction_entry.title, BODY_FONT_SIZE),
)

header_fragment = "Foreword Yet even in the auto industry"
print()
print(f"Header fragment in raw text   : {header_fragment in raw_text}")
print(f"Header fragment in clean text : {header_fragment in clean_text}")

raw_fragment_index = raw_text.index(header_fragment)
print()
print("Raw snippet around the leak:")
print(raw_text[raw_fragment_index - 25:raw_fragment_index + 120])

assert header_fragment in raw_text
assert header_fragment not in clean_text

Running-header page: 21
Raw top lines:
  top=  35.9 size=10 text='xx'
  top=  36.0 size=10 text='Foreword'
  top=  71.9 size=10 text='Yet even in the auto industry, the bulk of the work lies not in manufacturing but in'

After remove_running_header():
  top=  71.9 size=10 text='Yet even in the auto industry, the bulk of the work lies not in manufacturing but in'
  top=  83.9 size=10 text='maintenance—or its avoidance. In software, 80% or more of what we do is quaintly called'
  top=  95.9 size=10 text='“maintenance”: the act of repair. Rather than embracing the typical Western focus on pro-'

Header fragment in raw text   : True
Header fragment in clean text : False

Raw snippet around the leak:
nspire much of Scrum. xx Foreword Yet even in the auto industry, the bulk of the work lies not in manufacturing but in maintenance—or its avoidan


In [12]:
meaningful_names_entry = next(item for item in content_toc if item.title == "Chapter 2: Meaningful Names")
meaningful_lines = get_page_lines(doc, meaningful_names_entry.page)
meaningful_raw_lines = get_page_lines(doc, meaningful_names_entry.page, strip_running_header=False)

print(f"Chapter-opening page: {meaningful_names_entry.page}")
print(f"Visible lines with stripping   : {len(meaningful_lines)}")
print(f"Visible lines without stripping: {len(meaningful_raw_lines)}")
print()
for line in meaningful_lines[:5]:
    print(f"  top={line['top']:>6.1f} size={line['font_size']:>2} text={line['text']!r}")

assert any(line["text"] == "Meaningful Names" for line in meaningful_lines)
assert len(meaningful_lines) == len(meaningful_raw_lines)

Chapter-opening page: 48
Visible lines with stripping   : 8
Visible lines without stripping: 8

  top= 621.9 size=10 text='17'
  top= 102.4 size=48 text='2'
  top= 202.7 size=24 text='Meaningful Names'
  top= 238.9 size=10 text='by Tim Ottinger'
  top= 537.1 size=14 text='Introduction'


## Build embeddings and save retrieval artifacts

The final step encodes each kept chunk with `all-MiniLM-L6-v2`, stores normalized vectors in a FAISS `IndexFlatL2`, and writes parallel JSON metadata for retrieval.

Saved artifacts:
- `chunks.index` for vector search
- `chunks_metadata.json` for chunk text and section metadata

In [13]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [chunk.content for chunk in chunks]
embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

index = faiss.IndexIDMap(faiss.IndexFlatL2(embeddings.shape[1]))
ids = np.arange(len(embeddings), dtype=np.int64)
index.add_with_ids(embeddings, ids)
faiss.write_index(index, FAISS_INDEX_PATH)

metadata = [
    {
        "content": chunk.content,
        "page_number": chunk.page_number,
        "main_section": chunk.main_section,
        "section": chunk.section,
        "subsection": chunk.subsection,
        "chunk_index": chunk.chunk_index,
    }
    for chunk in chunks
]
with open(CHUNKS_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved FAISS index with {index.ntotal} chunks to {FAISS_INDEX_PATH}")
print(f"Saved chunk metadata with {len(metadata)} chunks to {CHUNKS_METADATA_PATH}")

INFO: Loading faiss with AVX512 support.
INFO: Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
INFO: Loading faiss with AVX2 support.
INFO: Successfully loaded faiss with AVX2 support.
d:\CodingProjects\SoftUniLLMProject\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
INFO: Use pytorch device_name: cpu
INFO: Load pretrained SentenceTransformer: all-MiniLM-L6-v2
Batches: 100%|██████████| 19/19 [00:14<00:00,  1.33it/s]

Saved FAISS index with 582 chunks to chunks.index
Saved chunk metadata with 582 chunks to chunks_metadata.json
